# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vikraamkumar-ds/flyrank-internship-ml/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Lane 2: Refresh / Content Opportunity Scoring.**

I'm picking this lane because the underlying question is the easiest one for me to reason about
honestly as a beginner: *out of thousands of content pages, which ones should a reviewer look at
first, this cycle, given they only have time for a few dozen?* That's a decision, not just a
prediction, and it's exactly the shape of question the `framing-ml-problems` skill asks for.

It also happens to be the lane the starter pipeline in this repo already demonstrates end to end
(`scripts/01`-`05`), so I can see, on real numbers, that a learned ranking beats a hand-written
rule *before* I commit seven weeks to it — see Section 3 below. Lanes 1 and 3 (signal analysis,
clustering) are more exploratory and don't end in an action a reviewer takes. Lane 4 (CTR/engagement
scoring) is a good candidate too, and I may fold some of its position-adjusted CTR ideas in as
features later, but "review queue for refresh" is the more complete, decision-shaped output to
start from.

I'm keeping this provisional — the guide gives me until the end of Week 4 to confirm or switch,
and I plan to revisit once I've done the signal audit (ML-06).

In [16]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("rows:", df.shape[0])
print("columns:", df.shape[1])
print("distinct clients:", df["client_id"].nunique())


rows: 30000
columns: 44
distinct clients: 32


## 2. The question: decision, action, cost of a wrong call

**Research question:** Which content pages should a content/SEO reviewer look at first this
review cycle, out of a much larger inventory, when their time only covers a small slice of it?

**Unit of analysis:** one content item (`content_id`) — one pseudonymized page, described by its
trailing-90-day search and engagement metrics.

**Decision it improves:** the order in which a limited-capacity human reviewer works through the
content backlog. Today that ordering can only really be done with a few hand-written rules (e.g.
"anything stale with traffic"); the question is whether a learned ranking orders that backlog
better than the rule does.

**Who acts, and what do they do:** a content strategist or SEO editor with a fixed weekly
capacity (say, the top 50 pages) opens the ranked queue, reads the reason codes attached to each
page (why it was flagged — stale, declining-with-demand, weak CTR for its position, etc.), and
decides an action per page: refresh, expand, protect, prune, or just monitor.

**Cost of a wrong call:** this is asymmetric, and both directions are real costs.
- *False positive* (flagged high, not actually worth it): the reviewer spends real hours refreshing
  or rewriting a page that wasn't declining or under-capturing clicks — wasted editor time, and a
  seat at the top of the queue taken from a page that needed it more.
- *False negative* (a genuinely declining, high-traffic page never surfaces): it keeps losing
  visibility silently, with nobody looking, until organic traffic/revenue has already been lost
  for weeks.

Because review capacity is fixed and small relative to the inventory, what matters most is the
precision of the *top* of the ranked list, not overall accuracy — which is why the starter
pipeline already tracks **precision@50** rather than a blanket accuracy score.

**Why data or ML can help at all:** a single if-statement rule (e.g. "stale AND high impressions")
is easy to write but blunt — it can't weigh several moving, correlated signals (position,
freshness, word count, engagement, trend) against each other, or learn how they trade off in this
data. That's a real pattern, but it's tangled enough that a plain rule leaves a lot of true
decliners unranked. That's the gap a learned ranking is meant to close — Section 3 shows the size
of that gap on the starter slice.

In [17]:
# How much of the current inventory a hand-written rule alone would flag as "worth reviewing" —
# this is the scale problem: too many candidates for a small reviewer capacity, which is why
# ORDER (ranking), not just a yes/no flag, is the real decision being improved.

declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
low_ctr_visible_page = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()

print(f"declining_with_demand candidates: {declining_with_demand:,} of {len(df):,} rows "
      f"({100*declining_with_demand/len(df):.1f}%)")
print(f"low_ctr_visible_page candidates: {low_ctr_visible_page:,} of {len(df):,} rows "
      f"({100*low_ctr_visible_page/len(df):.1f}%)")
print("A reviewer with capacity for ~50 pages per cycle cannot work through "
      "thousands of rule-flagged candidates without a ranking.")


declining_with_demand candidates: 13,152 of 30,000 rows (43.8%)
low_ctr_visible_page candidates: 9,759 of 30,000 rows (32.5%)
A reviewer with capacity for ~50 pages per cycle cannot work through thousands of rule-flagged candidates without a ranking.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [18]:
# Three real numbers that make this lane worth the next 7 weeks:

n = len(df)
declining_with_demand = ((df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)).sum()
low_ctr_visible_page = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
).sum()
declining_label_rate = (df["trend_direction"] == "down").mean()

print(f"1) {declining_with_demand:,} / {n:,} pages ({100*declining_with_demand/n:.1f}%) show "
      f"declining trend with real demand (>=100 impressions/90d) -- far too many for manual "
      f"review, so ranking order matters.")
print(f"2) {low_ctr_visible_page:,} / {n:,} pages ({100*low_ctr_visible_page/n:.1f}%) are visible "
      f"(position <=20, >=500 impressions) but under-capturing clicks for that position -- a "
      f"second, different opportunity signal worth folding into scoring.")
print(f"3) Baseline-rule score reaches only 0.240 precision@50, versus 0.740 precision@50 for the "
      f"random forest already trained in this repo's pipeline (outputs/model_report.md) -- "
      f"of the top 50 reviewer picks, the rule gets ~12 genuinely declining pages right, the "
      f"learned ranking gets ~37. That's the gap this lane is trying to close.")
print(f"(for reference: overall declining-label rate is {100*declining_label_rate:.1f}% across "
      f"{n:,} rows and {df['client_id'].nunique()} clients)")


1) 13,152 / 30,000 pages (43.8%) show declining trend with real demand (>=100 impressions/90d) -- far too many for manual review, so ranking order matters.
2) 9,759 / 30,000 pages (32.5%) are visible (position <=20, >=500 impressions) but under-capturing clicks for that position -- a second, different opportunity signal worth folding into scoring.
3) Baseline-rule score reaches only 0.240 precision@50, versus 0.740 precision@50 for the random forest already trained in this repo's pipeline (outputs/model_report.md) -- of the top 50 reviewer picks, the rule gets ~12 genuinely declining pages right, the learned ranking gets ~37. That's the gap this lane is trying to close.
(for reference: overall declining-label rate is 54.2% across 30,000 rows and 32 clients)


## 4. Careful words: what I can and can't claim

**What I can claim, by the end:**
- Observed associations between safe, pre-decision signals (position, freshness, CTR, engagement,
  word count, trend) and a page's ranking priority.
- A directional result: a learned ranking orders review candidates better than a transparent
  baseline rule, on precision@K, under proper client-holdout (and later, time-aware) validation.
- A decision-support output: a ranked queue with reason codes a human reviewer inspects and acts
  on — not an automatic decision.

**What I will never claim:**
- That a refresh *caused* a recovery — that needs an experiment or causal design, which this data
  alone can't give me.
- Anything about Google's actual ranking algorithm, or "AI ranking/citation" behavior.
- That the current `is_declining_label` (which is just `trend_direction == "down"`, computed from
  the *current* window) is the ideal target — it's a beginner proxy label, not a future outcome.
  I intend to move toward a future-window label (prior 90 days of features -> next 30 days
  decline) once I've done the signal audit, exactly as the lane guide recommends, so the model is
  predicting something that hasn't happened yet rather than describing something that already has.
- Any result beyond this 30,000-row, 32-client anonymized starter slice, without re-earning it on
  the full ~79M-row warehouse with its own leakage and panel checks.

In [19]:
# A first, small leakage discipline check: the label trap the flyrank-data skill warns about --
# trend_direction (and the trend_pct it's derived from) must never be used as a FEATURE, only as
# the label, since is_declining_label is itself derived from trend_direction.

candidate_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]

assert "trend_direction" not in candidate_features
assert "trend_pct" not in candidate_features
print("OK: label-derivation columns (trend_direction, trend_pct) are excluded from candidate features.")


OK: label-derivation columns (trend_direction, trend_pct) are excluded from candidate features.
